# Комбинация 9: Reflection + Ensemble — маркетинговый копирайтинг

Три генератора с разными стилями (формальный, разговорный, провокационный) параллельно пишут варианты текста. Каждый проходит через цикл рефлексии — генератор создаёт черновик, критик даёт замечания, генератор дорабатывает. После рефлексии мета-агент (креативный директор) выбирает лучший вариант или комбинирует сильные стороны всех трёх.

```mermaid
graph LR
    START((START)) -->|Send x3| reflection_variant

    subgraph reflection_variant ["Параллельные рефлексии (x3)"]
        generator(["🔹 Generator<br/><i>генерирует вариант</i>"]) -->|"черновик"| critic(["📊 Critic<br/><i>оценивает</i>"])
        critic -->|"замечания"| generator
    end

    reflection_variant --> meta_agent(["📊 Meta-agent<br/><i>выбирает лучший</i>"])
    meta_agent --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef subgraphStyle fill:#FDEBD0,stroke:#E67E22,stroke-width:2px,color:#333

    class START,END terminal
    class generator worker
    class critic,meta_agent aggregator
```

**Механизм:** `Send()` запускает три параллельных узла, каждый содержит внутренний цикл рефлексии (генератор–критик). Результаты собираются через аддитивный редьюсер и передаются мета-агенту для финального отбора.

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояния

`State` — основное состояние графа. Поле `polished_variants` использует редьюсер `operator.add`: каждый параллельный узел возвращает список из одного элемента, и LangGraph автоматически собирает их в общий список.

`AuthorState` — входное состояние для каждого параллельного узла. Содержит задачу, описание стиля и идентификатор автора. Это состояние передаётся через `Send()` и не пересекается между параллельными ветками.

In [3]:
llm = get_llm()

# --- Состояние для одного автора ---
class AuthorState(TypedDict):
    task: str
    style: str
    author_id: int

# --- Основное состояние ---
class State(TypedDict):
    task: str
    polished_variants: Annotated[list[str], operator.add]
    final_text: str

STYLES = [
    ("формальный", "Пиши в строгом деловом стиле. Факты, цифры, профессионализм."),
    ("разговорный", "Пиши живым, дружелюбным языком. Как будто рассказываешь другу."),
    ("провокационный", "Пиши смело, с вызовом. Провоцируй на размышление."),
]

## Fan-out: запуск параллельных авторов

Функция `spawn_authors` возвращает список из трёх объектов `Send()` — по одному на каждый стиль. LangGraph запустит все три узла параллельно, каждый со своим `AuthorState`. Это паттерн Ensemble: вместо одного генератора мы запускаем несколько с разными инструкциями и потом выбираем лучший результат.

In [4]:
def spawn_authors(state: State) -> list[Send]:
    """Fan-out: запуск трёх авторов с разными стилями параллельно."""
    print(f"  Запускаю {len(STYLES)} авторов с разными стилями")
    return [
        Send("author_with_reflection", {
            "task": state["task"], "style": desc, "author_id": i,
        })
        for i, (name, desc) in enumerate(STYLES)
    ]

## Автор с рефлексией

Каждый параллельный узел содержит внутренний цикл рефлексии: генерация черновика, критика, доработка. Для простоты реализации рефлексия встроена в функцию узла — это допустимо, когда число итераций мало и фиксировано.

Цикл внутри одного узла: копирайтер пишет черновик в заданном стиле, критик даёт замечания, копирайтер дорабатывает текст с учётом критики. Итого 3 вызова LLM на каждого автора.

In [5]:
def author_with_reflection(state: AuthorState) -> dict:
    """Один автор со встроенной рефлексией: черновик → критика → доработка."""
    style_name = STYLES[state["author_id"]][0]

    # Генерация черновика
    draft = llm.invoke([
        SystemMessage(content=f"Ты — копирайтер. Стиль: {state['style']}\nНапиши рекламный текст (3-4 предложения)."),
        HumanMessage(content=state["task"])
    ]).content

    # Критика
    critique = llm.invoke([
        SystemMessage(content="Оцени текст. Дай 1-2 конкретных замечания для улучшения."),
        HumanMessage(content=draft)
    ]).content

    # Доработка с учётом критики
    polished = llm.invoke([
        SystemMessage(content=f"Улучши текст по замечаниям. Сохрани стиль: {state['style']}"),
        HumanMessage(content=f"Текст:\n{draft}\n\nЗамечания:\n{critique}")
    ]).content

    print(f"  [{style_name}] Черновик → критика → финал ({len(polished)} символов)")
    return {"polished_variants": [f"[{style_name}]:\n{polished}"]}

## Мета-агент: выбор лучшего варианта

Мета-агент играет роль креативного директора. Он получает все три отшлифованных варианта (каждый уже прошёл рефлексию) и принимает решение: выбрать лучший или скомбинировать сильные стороны каждого в финальный текст.

In [6]:
def meta_agent(state: State) -> dict:
    """Мета-агент: выбирает лучший вариант или комбинирует сильные стороны."""
    all_variants = "\n\n---\n\n".join(state["polished_variants"])
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — креативный директор. Перед тобой 3 варианта текста "
            "в разных стилях, каждый прошёл через рефлексию. "
            "Выбери лучший ИЛИ создай финальную версию, "
            "комбинируя сильные стороны каждого."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\nВарианты:\n{all_variants}")
    ])
    print(f"  [Мета-агент] Финальный текст выбран/создан")
    return {"final_text": response.content}

## Сборка графа

Граф состоит из двух узлов: `author_with_reflection` (запускается параллельно через `Send()`) и `meta_agent`. Условное ребро из `START` вызывает `spawn_authors`, которая создаёт три параллельных экземпляра. После завершения всех рефлексий результаты собираются в `polished_variants` через аддитивный редьюсер, и управление переходит к мета-агенту.

In [7]:
graph = StateGraph(State)
graph.add_node("author_with_reflection", author_with_reflection)
graph.add_node("meta_agent", meta_agent)

graph.add_conditional_edges(START, spawn_authors)
graph.add_edge("author_with_reflection", "meta_agent")
graph.add_edge("meta_agent", END)

app = graph.compile()

## Запуск

Передаём задачу на генерацию рекламного текста. Три автора параллельно пройдут цикл рефлексии (черновик, критика, доработка — по 3 вызова LLM каждый), а затем мета-агент выберет или скомбинирует лучший результат. Итого до 10 вызовов LLM.

In [8]:
result = app.invoke({
    "task": "Рекламный текст для платформы мультиагентных систем для бизнеса",
    "polished_variants": [],
    "final_text": "",
})

print(f"\nВариантов собрано: {len(result['polished_variants'])}")
print(f"\n{'=' * 60}")
print("ФИНАЛЬНЫЙ ТЕКСТ:")
print("=" * 60)
print(result["final_text"])

  Запускаю 3 авторов с разными стилями


  [формальный] Черновик → критика → финал (716 символов)


  [провокационный] Черновик → критика → финал (674 символов)
  [разговорный] Черновик → критика → финал (809 символов)


  [Мета-агент] Финальный текст выбран/создан

Вариантов собрано: 3

ФИНАЛЬНЫЙ ТЕКСТ:
Вот финальная версия — беру лучшее из всех трёх: ясность формального, динамику разговорного и энергию провокационного, но без перегиба.

**Платформа мультиагентных систем для бизнеса помогает автоматизировать рутину, ускорить обработку данных и выстроить более управляемые бизнес-процессы.**  
Несколько интеллектуальных агентов могут работать вместе: один собирает и анализирует информацию, другой распределяет задачи, третий контролирует статусы и помогает согласовывать действия между подразделениями.  
**В результате команда меньше тратит времени на ручные операции, снижается нагрузка на сотрудников, а управление становится прозрачнее и быстрее.**  
Платформа подходит компаниям, где важны высокая операционная скорость, точность данных и стабильная работа процессов без роста административной нагрузки.  

**Автоматизируйте то, что тормозит бизнес, и освободите команду для задач, которые действительно двиг

## Итоги

**Reflection + Ensemble** объединяет два паттерна для творческих задач, где важны и качество, и разнообразие:

- **Ensemble** (через `Send()`) запускает три генератора параллельно с разными стилевыми инструкциями — формальным, разговорным и провокационным. Это даёт разнообразие вариантов.
- **Reflection** (встроенный цикл генератор-критик) повышает качество каждого варианта до передачи мета-агенту. Каждый автор проходит черновик, критику и доработку.
- **Мета-агент** выступает финальным арбитром: выбирает лучший вариант или комбинирует сильные стороны всех трёх.

**Стоимость:** ~10 вызовов LLM (3 автора x 3 вызова + 1 мета-агент). Для маркетингового копирайтинга, где один текст может стоить десятки тысяч рублей, это оправданные затраты. Три стиля плюс рефлексия дают разнообразие и качество одновременно.